In [1]:
import os
import torch

# Set GPU index (z. B. 1 für die zweite GPU)
os.environ["CUDA_VISIBLE_DEVICES"] = "2,5,6,7"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

# Dein Prompt-Template
def build_prompt(text):
    return (
        f"Statement: {text}\n"
        f"Task: Determine whether the statement is factually correct or fake news. Use only publicly verifiable and objective facts. Do not guess.\n"
        f"Answer with only one word: True or False.\n"
        f"Answer:"
    )


# Modell
model_name = "meta-llama/Meta-Llama-3-70B-Instruct"

print(f"Lade Modell: {model_name}")

# Lade Tokenizer und Modell (mit automatischer GPU-Verteilung)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # verteilt automatisch auf verfügbare GPUs
    torch_dtype=torch.float16,  # effizienter für große Modelle
)
model.eval()

# CSV laden
df = pd.read_csv("preprocessed_test_cleaned.csv", sep="\t").iloc[:2000].copy()
statements = df["statement"].tolist()

# Inferenz
results = []
for text in statements:
    prompt = build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=5)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    result = decoded.split("Answer:")[-1].strip().split()[0]
    results.append(result)

# Speichern
df["label_llama3_70b"] = results
# df.to_csv("statements_with_predictions.csv", index=False)
print(df.head())


Lade Modell: meta-llama/Meta-Llama-3-70B-Instruct


2025-07-01 17:47:03.960819: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751384823.983715    5024 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751384823.989580    5024 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-01 17:47:04.010989: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/pytorch/aten/src/ATen/native/cuda/TensorCompare.cu:110: _assert_async_cuda_kernel: block: [0,0,0], thread: [0,0,0] Assertion `probability tensor contains either `inf`, `nan` or element < 0` failed.


KeyboardInterrupt: 

In [ ]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# Definiere deine Labelgruppen
true_labels = ["true", "mostly-true", "half-true"]
false_labels = ["barely-true", "false", "pants-fire", "pants-on-fire"]

# Mapping-Funktion: Normalisiert die Labels zu "True" oder "False"
def map_label(label):
    label = str(label).strip().lower()
    if label in true_labels:
        return "True"
    elif label in false_labels:
        return "False"
    else:
        return "False"  # Default für unbekannte Labels

# Wandle Ground Truth und Prediction in binäre Kategorien um
y_true = df["label"].apply(map_label)
y_pred = df["label_Llama-3.1-8B-Instruct"].apply(lambda x: map_label(x))

# Konfusionsmatrix
cm = confusion_matrix(y_true, y_pred, labels=["True", "False"])
print("Confusion Matrix (True/False):")
print(cm)

print(f"\nAccuracy:  {accuracy_score(y_true, y_pred)}")
print(f"Precision: {precision_score(y_true, y_pred, pos_label='True')}  (bezogen auf 'True')")
print(f"Recall:    {recall_score(y_true, y_pred, pos_label='True')}  (bezogen auf 'True')")
print(f"F1 Score:  {f1_score(y_true, y_pred, pos_label='True')}  (bezogen auf 'True')")

# Optionaler vollständiger Bericht
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["True", "False"]))
